# Escalation Ladder — LaTeX table version

Simple table from `config/paper_labels.yaml`. Edit the YAML to change content.

**Output:** `latex/tables/escalation_ladder.tex`

In [1]:
import sys
import os
from pathlib import Path

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
OUT = ROOT / 'outputs' / 'tables' / 'escalation_ladder.tex'

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from analysis.labels import TIER_LADDER, TIER_META

In [2]:
# Build tier rows from YAML
tiers = []
for _, slug, short, full in TIER_LADDER:
    m = TIER_META[slug]
    tiers.append({
        'num':   m['number'],
        'name':  m.get('full', short),
        'desc':  m.get('description', ''),
        'group': m.get('group', 'context'),
    })

n_ctx  = sum(1 for t in tiers if t['group'] == 'context')
n_pres = sum(1 for t in tiers if t['group'] == 'pressure')

lines = []
lines.append('\\begin{table}[t]')
lines.append('\\centering')
lines.append('\\small')
lines.append('\\begin{tabular}{clll}')
lines.append('\\toprule')
lines.append('\\textbf{Tier} & \\textbf{Name} & \\textbf{Prompt content} & \\textbf{Condition type} \\\\')
lines.append('\\midrule')

ctx_done = False
ctx_i = 0
pres_i = 0
for t in tiers:
    desc = t['desc'].replace('*', '{*}')
    if t['group'] == 'context':
        if ctx_i == 0:
            right = f'\\multirow{{{n_ctx}}}{{*}}{{Occupational details}}'
        else:
            right = ''
        ctx_i += 1
    else:
        if not ctx_done:
            lines.append('\\midrule')
            ctx_done = True
        if pres_i == 0:
            right = f'\\multirow{{{n_pres}}}{{*}}{{Organizational pressure}}'
        else:
            right = ''
        pres_i += 1
    lines.append(f"{t['num']} & {t['name']} & {desc} & {right} \\\\")

lines.append('\\bottomrule')
lines.append('\\end{tabular}')
lines.append('\\caption{Contextual escalation ladder. Each tier is cumulative, adding to all prior tiers. Tiers 4--7 constitute organizational pressure conditions.}')
lines.append('\\label{tab:escalation}')
lines.append('\\end{table}')

tex = '\n'.join(lines) + '\n'
OUT.write_text(tex)
print(f'Wrote {OUT}')
print()
print(tex)

Wrote ./latex/tables/escalation_ladder.tex

\begin{table}[t]
\centering
\small
\begin{tabular}{clll}
\toprule
\textbf{Tier} & \textbf{Name} & \textbf{Prompt content} & \textbf{Condition type} \\
\midrule
1 & Baseline & Job title only & \multirow{3}{*}{Occupational details} \\
2 & Job duties & + Six O{*}NET key responsibilities &  \\
3 & Job context & + Dimension-specific job requirements &  \\
\midrule
4 & Hiring manager memo & Tiers 1--3 + personal disclosure pressure & \multirow{4}{*}{Organizational pressure} \\
5 & Senior operations memo & Tiers 1--3 + seniority-based authority override &  \\
6 & Policy memo & Tiers 1--3 + simulated company policy &  \\
7 & Insurance memo & Tiers 1--3 + simulated insurance mandate &  \\
\bottomrule
\end{tabular}
\caption{Contextual escalation ladder. Each tier is cumulative, adding to all prior tiers. Tiers 4--7 constitute organizational pressure conditions.}
\label{tab:escalation}
\end{table}

